# Phase III Kappa Parameter Study (Accelerated SGS, 65 x 65)

This notebook repeats the `κ` (`rkappa`) parameter study using the accelerated Symmetric Gauss-Seidel sweep functions:

- `SGS_forward_sweep_acc`
- `SGS_backward_sweep_acc`

This is the report-quality `65 x 65` duplicate of the faster `33 x 33` screening notebook. It keeps the narrowed kappa range from the screening run and restores the stricter convergence tolerance.

In [ ]:
from pathlib import Path
import contextlib
import io
import os
import shutil
import time

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name != "start-code" and (Path.cwd() / "start-code").exists():
    os.chdir(Path.cwd() / "start-code")

plot_dir = Path("Phase III Kappa Study Accelerated 65x65")
plot_dir.mkdir(exist_ok=True)

print(f"Working directory: {Path.cwd()}")
print(f"Plot output directory: {plot_dir.resolve()}")

In [ ]:
# Report-quality settings based on the faster 33x33 screening pass.
NODE_COUNT = 65
REYNOLDS_NUMBER = 10.0
CFL = 0.5
TOLERANCE = 1e-10

# Narrowed range from the screening run. Add more values if the 65x65 results suggest a different optimum.
KAPPA_VALUES = [0.25, 0.5, 1.0]

# Run one tiny accelerated case first so the Numba compile cost is paid before timing the study cases.
WARM_UP_ACCELERATED_SGS = True

pd.DataFrame({
    "nodes": [f"{NODE_COUNT}x{NODE_COUNT}"] * len(KAPPA_VALUES),
    "Re": [REYNOLDS_NUMBER] * len(KAPPA_VALUES),
    "CFL": [CFL] * len(KAPPA_VALUES),
    "tolerance": [TOLERANCE] * len(KAPPA_VALUES),
    "kappa": KAPPA_VALUES,
})

In [ ]:
MAIN_PLOT_FILES = [
    "ucontour.png",
    "vcontour.png",
    "pcontour.png",
    "residualcomponent.png",
    "residual.png",
]


def replace_once(source, old, new, solver_path):
    if old not in source:
        raise RuntimeError(f"Could not find expected setting in {solver_path}: {old}")
    return source.replace(old, new, 1)


def format_solver_float(value):
    return f"{value:<16g}"


def patched_solver_source(kappa, node_count=NODE_COUNT, tolerance=TOLERANCE, max_iterations=None):
    solver_path = Path("main_solver.py")
    source = solver_path.read_text()

    replacements = {
        "imax = 9               # Number of points in the x-direction (use odd numbers only)":
            f"imax = {node_count:<15}# Number of points in the x-direction (use odd numbers only)",
        "jmax = 9               # Number of points in the y-direction (use odd numbers only)":
            f"jmax = {node_count:<15}# Number of points in the y-direction (use odd numbers only)",
        "iterout = 5000         # Number of time steps between solution output":
            "iterout = 100000000    # Number of time steps between solution output",
        "cfl = 0.5              # CFL number used to determine time step":
            f"cfl = {format_solver_float(CFL)}# CFL number used to determine time step",
        "toler = 1e-10          # Tolerance for iterative residual convergence":
            f"toler = {format_solver_float(tolerance)}# Tolerance for iterative residual convergence",
        "rkappa = 0.5           # Time derivative preconditioning constant":
            f"rkappa = {format_solver_float(kappa)}# Time derivative preconditioning constant",
        "Re = 10.0              # Reynolds number = rho*Uinf*L/rmu":
            f"Re = {format_solver_float(REYNOLDS_NUMBER)}# Reynolds number = rho*Uinf*L/rmu",
        "vectorize = False":
            "vectorize = True",
        "u = SGS_forward_sweep(u, uold, dt, s, rho, rhoinv, dx, dy, rkappa, rmu, vel2ref, artviscx, artviscy)":
            "u = SGS_forward_sweep_acc(u, uold, dt, s, rho, rhoinv, dx, dy, rkappa, rmu, vel2ref, artviscx, artviscy)",
        "u = SGS_backward_sweep(u, uold, dt, s, rho, rhoinv, dx, dy, rkappa, rmu, vel2ref, artviscx, artviscy)":
            "u = SGS_backward_sweep_acc(u, uold, dt, s, rho, rhoinv, dx, dy, rkappa, rmu, vel2ref, artviscx, artviscy)",
    }

    if max_iterations is not None:
        replacements["nmax = 500000          # Maximum number of iterations"] = (
            f"nmax = {max_iterations:<15}# Maximum number of iterations"
        )

    for old, new in replacements.items():
        source = replace_once(source, old, new, solver_path)

    return solver_path, source


def archive_solver_plots(case_label):
    for filename in MAIN_PLOT_FILES:
        src = Path(filename)
        if src.exists():
            dst = plot_dir / f"{case_label}_{filename}"
            if dst.exists():
                dst.unlink()
            shutil.move(str(src), str(dst))


def run_kappa_case(kappa, node_count=NODE_COUNT, tolerance=TOLERANCE, archive_plots=True, max_iterations=None):
    case_label = f"kappa_{kappa:g}".replace(".", "p")
    solver_path, source = patched_solver_source(
        kappa,
        node_count=node_count,
        tolerance=tolerance,
        max_iterations=max_iterations,
    )
    namespace = {
        "__file__": str(solver_path.resolve()),
        "__name__": "__main__",
    }

    print(f"Running accelerated SGS: {node_count}x{node_count}, Re={REYNOLDS_NUMBER:g}, kappa={kappa:g}")
    start = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()) as captured_output:
        exec(compile(source, str(solver_path), "exec"), namespace)
    elapsed = time.perf_counter() - start

    if archive_plots:
        archive_solver_plots(case_label)
    plt.close("all")

    u = namespace["u"].copy()
    imax, jmax, _ = u.shape
    i_mid = (imax - 1) // 2
    j_mid = (jmax - 1) // 2
    x_coords = np.linspace(namespace["xmin"], namespace["xmax"], imax)
    y_coords = np.linspace(namespace["ymin"], namespace["ymax"], jmax)

    return {
        "kappa": kappa,
        "label": case_label,
        "nodes": node_count,
        "Re": REYNOLDS_NUMBER,
        "CFL": CFL,
        "tolerance": tolerance,
        "elapsed_time_sec": elapsed,
        "iterations": int(namespace["n"]),
        "converged": bool(namespace["isConverged"]),
        "final_conv": float(namespace["conv"]),
        "final_residual": np.array(namespace["res"], copy=True),
        "conv_history": np.array(namespace["convVector"], copy=True),
        "residual_history": np.array(namespace["resPMatrix"], copy=True),
        "x": x_coords,
        "y": y_coords,
        "u_vertical_centerline": u[i_mid, :, 1].copy(),
        "v_horizontal_centerline": u[:, j_mid, 2].copy(),
        "p_vertical_centerline": u[i_mid, :, 0].copy(),
        "p_horizontal_centerline": u[:, j_mid, 0].copy(),
        "captured_output": captured_output.getvalue(),
    }

In [ ]:
if WARM_UP_ACCELERATED_SGS:
    print("Warming up accelerated SGS kernels on a small 9x9 case...")
    _warmup = run_kappa_case(
        0.5,
        node_count=9,
        tolerance=1e-3,
        archive_plots=False,
        max_iterations=50,
    )
    print(f"Warm-up complete in {_warmup['elapsed_time_sec']:.2f} s")

results = []
for kappa in KAPPA_VALUES:
    result = run_kappa_case(kappa)
    results.append(result)
    print(
        f"Finished kappa={kappa:g}: "
        f"iterations={result['iterations']}, "
        f"converged={result['converged']}, "
        f"final_conv={result['final_conv']:.3e}, "
        f"wall_time={result['elapsed_time_sec']:.2f} s"
    )

summary_table = pd.DataFrame([
    {
        "kappa": result["kappa"],
        "nodes": f"{result['nodes']}x{result['nodes']}",
        "Re": result["Re"],
        "CFL": result["CFL"],
        "tolerance": result["tolerance"],
        "iterations": result["iterations"],
        "converged": result["converged"],
        "final_conv": result["final_conv"],
        "wall_time_s": result["elapsed_time_sec"],
        "continuity_residual": result["final_residual"][0],
        "x_momentum_residual": result["final_residual"][1],
        "y_momentum_residual": result["final_residual"][2],
    }
    for result in results
]).sort_values("kappa")

summary_table

In [ ]:
fontsize = 12

plt.figure(figsize=(7, 5))
for result in results:
    history = np.maximum(result["conv_history"], 1e-300)
    plt.semilogy(history, linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("iteration", fontsize=fontsize)
plt.ylabel("overall residual", fontsize=fontsize)
plt.title(f"Accelerated SGS residual convergence on {NODE_COUNT}x{NODE_COUNT} mesh", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_accelerated_residual_histories.png", dpi=300)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(summary_table["kappa"], summary_table["iterations"], "o-", linewidth=2)
axes[0].set_xscale("log")
axes[0].set_xlabel("κ", fontsize=fontsize)
axes[0].set_ylabel("iterations", fontsize=fontsize)
axes[0].set_title("Iterations to convergence", fontsize=fontsize)

axes[1].plot(summary_table["kappa"], summary_table["wall_time_s"], "o-", linewidth=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("κ", fontsize=fontsize)
axes[1].set_ylabel("wall time (s)", fontsize=fontsize)
axes[1].set_title("Accelerated wall time to convergence", fontsize=fontsize)

for ax in axes:
    ax.tick_params(labelsize=fontsize)
    ax.grid(True, which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(plot_dir / "kappa_accelerated_efficiency_summary.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for result in results:
    plt.plot(result["u_vertical_centerline"], result["y"], linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("u velocity (m/s)", fontsize=fontsize)
plt.ylabel("y (m)", fontsize=fontsize)
plt.title("Vertical centerline u velocity", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_accelerated_vertical_centerline_u.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
for result in results:
    plt.plot(result["x"], result["v_horizontal_centerline"], linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("x (m)", fontsize=fontsize)
plt.ylabel("v velocity (m/s)", fontsize=fontsize)
plt.title("Horizontal centerline v velocity", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_accelerated_horizontal_centerline_v.png", dpi=300)
plt.show()

plt.figure(figsize=(7, 5))
for result in results:
    plt.plot(result["p_vertical_centerline"], result["y"], linewidth=2, label=f"κ={result['kappa']:g}")
plt.xlabel("p (N/m^2)", fontsize=fontsize)
plt.ylabel("y (m)", fontsize=fontsize)
plt.title("Vertical centerline pressure", fontsize=fontsize)
plt.legend(loc="best", fontsize=fontsize - 1, frameon=False)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.tight_layout()
plt.savefig(plot_dir / "kappa_accelerated_vertical_centerline_pressure.png", dpi=300)
plt.show()